# Start here

Fresh and Good is a fictional online grocery in Bangkok. The question today is which customers will place no paid order in the next 30 days. Yes means the team might send a small voucher. No means they leave the customer alone.

You will build that model on your own laptop. The data is synthetic. The score is for the classroom demo, not for a real shop.

## Objectives

- Frame the churn question and the voucher decision.
- See the path from framing through the form.
- Warm up with short Python, pandas, and scikit-learn exercises you will reuse later.

## What we will do

1. Finish three Python exercises: a customer dict, an f-string formatter, and a quiet-customer check.
2. Finish three pandas exercises on a tiny table you build in the notebook.
3. Finish four scikit-learn exercises: a train/test split, a tiny classifier, a tiny regressor, and a scaler.
4. Move on to the Look at the customers part to open the real customer table.

## The loop

1. Frame the question.
2. Look at the table.
3. Split the rows and score a majority baseline.
4. Preprocess, cross-validate, then engineer features and keep a short list.
5. Tune, then choose one churn model from the validation scores.
6. Choose the voucher threshold on the validation rows.
7. Score the churn test rows once.
8. Estimate next-month spend in the spend estimation part.
9. Show one customer in the form.

The README diagram shows which notebook does each step.

Work in groups. The group owns the cost of a wasted voucher and the cost of a missed churner. Those two numbers choose the threshold. At the end, compare scores so everyone can help pick the best model.


## Python sprint

A customer can be a dict. The keys are column names. The values are that customer's numbers and labels.

Run the example, then replace each exercise.


In [40]:
customer = {
    "plan": "monthly",
    "orders_90d": 6,
    "days_since_last_order": 4,
}
print(customer["plan"])
print(f"{customer['orders_90d']} orders")


monthly
6 orders


### Exercise 1

Build a dict named `soda_shopper`. Use `plan` `"none"`, `orders_90d` `1`, and `days_since_last_order` `40`.

Steps:

1. Create a dict with those three keys.
2. Assign it to `soda_shopper`.


In [41]:
soda_shopper = {
    "plan": "none",
    "orders_90d": 1,
    "days_since_last_order": 40,
}


In [42]:
if soda_shopper is None:
    raise NotImplementedError("Build the soda_shopper dict.")
assert soda_shopper["plan"] == "none"
assert soda_shopper["orders_90d"] == 1
assert soda_shopper["days_since_last_order"] == 40
print(soda_shopper)


{'plan': 'none', 'orders_90d': 1, 'days_since_last_order': 40}


### Exercise 2

`format_customer` returns one sentence. Use an f-string. The sentence for a weekly plan with 3 orders and a last order 2 days ago is `weekly plan, 3 orders, last order 2 days ago`.

Steps:

1. Read `plan`, `orders_90d`, and `days_since_last_order` from `row`.
2. Return one f-string in the exact shape shown above.


In [43]:
def format_customer(row):
    return (
        f"{row['plan']} plan, {row['orders_90d']} orders, last order {row['days_since_last_order']} days ago"
    )


In [44]:
sample = {"plan": "weekly", "orders_90d": 3, "days_since_last_order": 2}
assert format_customer(sample) == "weekly plan, 3 orders, last order 2 days ago"
print(format_customer(sample))


weekly plan, 3 orders, last order 2 days ago


### Exercise 3

`quiet_customer` returns True when `days_since_last_order` is greater than `cutoff`. Otherwise it returns False.

Steps:

1. Compare `days_since_last_order` to `cutoff`.
2. Return `True` only when the days value is greater than the cutoff.


In [53]:
def quiet_customer(days_since_last_order, cutoff):
    return days_since_last_order > cutoff


In [54]:
assert quiet_customer(40, 21) is True
assert quiet_customer(10, 21) is False
print("quiet", quiet_customer(40, 21))


quiet True


## Pandas sprint

Later notebooks load a CSV. Here you only need a tiny table built from a dict of lists. No file.

Run the example, then replace each exercise.


In [56]:
import pandas as pd

toy = pd.DataFrame(
    {
        "plan": ["weekly", "monthly", "weekly"],
        "orders_90d": [3, 8, 1],
        "days_since_last_order": [2, 5, 30],
    }
)
print(toy.shape)
toy


(3, 3)


,plan,orders_90d,days_since_last_order
0,weekly,3,2
1,monthly,8,5
2,weekly,1,30


### Exercise 4

Build a DataFrame named `aisle_rows` with three columns: `plan`, `orders_90d`, and `days_since_last_order`.

Use these three rows, in this order:

1. weekly, 4 orders, 1 day ago
2. monthly, 2 orders, 12 days ago
3. weekly, 7 orders, 3 days ago

Docs: [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)


In [62]:
aisle_rows = pd.DataFrame(
    {
        "plan": ["weekly", "monthly", "weekly"],
        "orders_90d": [4, 2, 7],
        "days_since_last_order": [1, 12, 3],
    }
)


In [63]:
if aisle_rows is None:
    raise NotImplementedError("Build the aisle_rows DataFrame.")
assert list(aisle_rows.columns) == ["plan", "orders_90d", "days_since_last_order"]
assert len(aisle_rows) == 3
assert list(aisle_rows["plan"]) == ["weekly", "monthly", "weekly"]
assert list(aisle_rows["orders_90d"]) == [4, 2, 7]
assert list(aisle_rows["days_since_last_order"]) == [1, 12, 3]
print(aisle_rows)


      plan  orders_90d  days_since_last_order
0   weekly           4                      1
1  monthly           2                     12
2   weekly           7                      3


### Exercise 5

Keep the rows in `aisle_rows` whose `orders_90d` is at least 4. Store them in `busy_aisle`.

Steps:

1. Filter `aisle_rows` with a boolean mask on `orders_90d >= 4`.
2. Assign the filtered table to `busy_aisle`.

Docs: [Boolean indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)


In [67]:
# marked = aisle_rows['orders_90d'] >= 4
# busy_aisle = aisle_rows[marked]

# มักเขียนอยู่ในรูปนี้มากกว่า
busy_aisle = aisle_rows[aisle_rows['orders_90d'] >= 4]

In [ ]:
if busy_aisle is None:
    raise NotImplementedError("Filter aisle_rows into busy_aisle.")
assert len(busy_aisle) == 2
assert busy_aisle["orders_90d"].min() >= 4
print(busy_aisle)


### Exercise 6

`mean_recency` is the mean of `days_since_last_order` on `aisle_rows`. Use pandas. The value is a float.

Steps:

1. Select the `days_since_last_order` column on `aisle_rows`.
2. Take its mean and store that number in `mean_recency`.

Docs: [Series.mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html)


In [68]:
mean_recency = aisle_rows["days_since_last_order"].mean()


In [69]:
if mean_recency is None:
    raise NotImplementedError("Compute mean_recency from aisle_rows.")
assert abs(float(mean_recency) - ((1 + 12 + 3) / 3)) < 1e-9
print("mean_recency", float(mean_recency))


mean_recency 5.333333333333333


## scikit-learn sprint

Same idea: tiny numbers, no CSV. You will split rows, fit one classifier, fit one regressor, then scale columns. Scaling is a preprocessing step later notebooks put inside pipelines.

Run the example, then replace each exercise.


In [98]:
from sklearn.model_selection import train_test_split

X_demo = [[1.0], [2.0], [3.0], [4.0]]
y_demo = [0, 0, 1, 1]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_demo, y_demo, test_size=0.5, random_state=0
)
print("train", len(X_tr), "test", len(X_te))
print("y_train", y_tr)


train 2 test 2
y_train [0, 0]


### Exercise 7

Call `train_test_split` on `X_tiny` and `y_tiny`. Use `test_size=0.25` and `random_state=42`. Store the four return values as `X_train`, `X_test`, `y_train`, and `y_test`.

Steps:

1. Import is already done in the example above if you ran it.
2. Call `train_test_split(X_tiny, y_tiny, test_size=0.25, random_state=42)`.
3. Unpack into `X_train`, `X_test`, `y_train`, `y_test` in that order.

Docs: [train_test_split](https://scikit-learn.org/1.9/modules/generated/sklearn.model_selection.train_test_split.html)


In [104]:
X_tiny = [[0.0], [1.0], [2.0], [3.0]]
y_tiny = [0, 0, 1, 1]
X_train, X_test, y_train, y_test = train_test_split(
    X_tiny, y_tiny, test_size=0.25, random_state=42
)


In [105]:
if X_train is None:
    raise NotImplementedError("Split X_tiny and y_tiny.")
assert len(X_train) == 3 and len(X_test) == 1
assert len(y_train) == 3 and len(y_test) == 1
print("train", len(X_train), "test", len(X_test), "y_test", y_test)


train 3 test 1 y_test [0]


### Exercise 8

Fit a `LogisticRegression` on `X_fit` and `y_fit`. Predict the class for `[[1.5]]` and store it in `pred` as a single integer (0 or 1).

Steps:

1. Create a `LogisticRegression` and `fit` it on `X_fit` and `y_fit`.
2. Call `predict` on `[[1.5]]`.
3. Store the single predicted class in `pred` as an int.

Docs: [LogisticRegression](https://scikit-learn.org/1.9/modules/generated/sklearn.linear_model.LogisticRegression.html)


In [ ]:
from sklearn.linear_model import LogisticRegression

X_fit = [[0.0], [1.0], [2.0], [3.0]]
y_fit = [0, 0, 1, 1]
pred = None


In [ ]:
if pred is None:
    raise NotImplementedError("Fit LogisticRegression and predict [[1.5]].")
pred = int(pred)
assert pred == 1
print("pred", pred)


### Exercise 9

Fit a `Ridge` regressor on `X_spend` and `y_spend`. Predict the amount for `[[2.5]]` and store it in `spend_pred` as a float.

Steps:

1. Create a `Ridge` and `fit` it on `X_spend` and `y_spend`.
2. Call `predict` on `[[2.5]]`.
3. Store the single predicted amount in `spend_pred` as a float.

Docs: [Ridge](https://scikit-learn.org/1.9/modules/generated/sklearn.linear_model.Ridge.html)


In [ ]:
from sklearn.linear_model import Ridge

X_spend = [[1.0], [2.0], [3.0], [4.0]]
y_spend = [100.0, 200.0, 300.0, 400.0]
spend_pred = None

In [ ]:
if spend_pred is None:
    raise NotImplementedError("Fit Ridge and predict [[2.5]].")
spend_pred = float(spend_pred)
assert abs(spend_pred - 250.0) < 1e-6
print("spend_pred", spend_pred)

### Exercise 10

Scale the feature columns before the model sees them. Columns like `orders_90d` and `days_since_last_order` live on different ranges. Without scaling, models that care about distance or gradient steps (logistic regression, SVM, ridge) overweight the large-range columns. `StandardScaler` puts each column near mean 0 and variance 1 so they compete on equal footing.

Fit a `StandardScaler` on `X_scale`, then transform the same rows. Store the scaled array in `X_scaled`. After scaling, each column mean should be about 0.

Docs: [StandardScaler](https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.StandardScaler.html)


In [ ]:
from sklearn.preprocessing import StandardScaler

X_scale = [[10.0, 1.0], [20.0, 3.0], [30.0, 5.0]]
X_scaled = None


In [ ]:
import numpy as np

if X_scaled is None:
    raise NotImplementedError("Fit StandardScaler and transform X_scale.")
X_scaled = np.asarray(X_scaled, dtype=float)
assert X_scaled.shape == (3, 2)
assert np.allclose(X_scaled.mean(axis=0), 0.0)
print(X_scaled)
